In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# RMI Tutorial 03: BigQuery Data Verification

Once your routes are active and historical data begins to accumulate, you can access this information directly in BigQuery via your **Linked Dataset**.

### Objectives:
1. **Locate** your RMI linked dataset.
2. **Verify** your registered routes in the `routes_status` table.
3. **Query** historical traffic data using BigQuery magics.
4. **Analyze** performance metrics like the Travel Time Index (TTI).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/tutorials/03_bq_verification.ipynb) 
[![Open In Colab Enterprise](https://img.shields.io/badge/Open%20in-Colab%20Enterprise-blue?logo=google-cloud&logoColor=white)](https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Ftutorials%2F03_bq_verification.ipynb) 
[![Open In BigQuery Notebooks](https://img.shields.io/badge/Open%20in-BigQuery%20Notebooks-blue?logo=google-cloud&logoColor=white)](https://console.cloud.google.com/bigquery/import?url=https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Ftutorials%2F03_bq_verification.ipynb) 
[![View on GitHub](https://img.shields.io/badge/View%20on-GitHub-lightgrey?logo=github&logoColor=white)](https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/tutorials/03_bq_verification.ipynb)


In [ ]:
from google.colab import auth
auth.authenticate_user()

## 1. Environment Setup
Set your Project ID and the standard RMI dataset name.

In [ ]:
import os
PROJECT_ID = 'YOUR_PROJECT_ID' # @param {type:"string"}
DATASET_ID = 'historical_roads_data' # @param {type:"string"}
os.environ['PROJECT_ID'] = PROJECT_ID
os.environ['DATASET_ID'] = DATASET_ID

## 2. Check Route Status
The `routes_status` table contains your route definitions and their current RMI state. We use the `%%bigquery` magic for direct SQL execution.

In [ ]:
%%bigquery --project $PROJECT_ID
SELECT 
  selected_route_id, 
  status, 
  validation_error 
FROM `historical_roads_data.routes_status` 
LIMIT 10

## 3. Query Historical Performance
Using the **Gold Standard** pattern: temporal filtering, geometry type checks, and joining with status for attributes.

In [ ]:
%%bigquery --project $PROJECT_ID
DECLARE start_date DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR);

WITH base_history AS (
  SELECT
    h.selected_route_id,
    h.record_time,
    h.duration_in_seconds,
    h.static_duration_in_seconds,
    s.route_attributes
  FROM `historical_roads_data.historical_travel_time` h
  JOIN `historical_roads_data.routes_status` s
    ON h.selected_route_id = s.selected_route_id
  WHERE h.record_time >= start_date
    -- Early Path Integrity Filter
    AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
)
SELECT
  *,
  -- Metric Calculation (Travel Time Index)
  SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) as travel_time_index
FROM base_history
ORDER BY record_time DESC
LIMIT 10

## 4. Visualize Traffic Trends
Finally, let's plot the **Travel Time Index (Ratio)** over time to see congestion patterns. We'll use `plotly` for a quick interactive chart.

In [ ]:
%%bigquery df_trends --project $PROJECT_ID
DECLARE start_date DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR);

SELECT
  record_time,
  selected_route_id,
  SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) as travel_time_index
FROM `historical_roads_data.historical_travel_time` 
WHERE record_time >= start_date
ORDER BY record_time ASC

In [ ]:
import plotly.express as px

fig = px.line(df_trends, x='record_time', y='travel_time_index', color='selected_route_id',
              title='Travel Time Index Over Last 24 Hours',
              labels={'travel_time_index': 'Congestion Ratio (Actual/Static)', 'record_time': 'Time (UTC)'})
fig.show()

## 5. Weekly Congestion Heatmap
Let's look at long-term trends by analyzing the average congestion ratio by **Hour of Day** and **Day of Week** over the last 30 days.

In [ ]:
%%bigquery df_heatmap --project $PROJECT_ID
DECLARE start_date DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY);

SELECT
  EXTRACT(DAYOFWEEK FROM record_time) as day_of_week,
  EXTRACT(HOUR FROM record_time) as hour_of_day,
  AVG(SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds)) as avg_tti
FROM `historical_roads_data.historical_travel_time` 
WHERE record_time >= start_date
GROUP BY 1, 2
ORDER BY 1, 2

In [ ]:
import plotly.graph_objects as go

days = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

# Pivot the data for the heatmap
pivot_df = df_heatmap.pivot(index='day_of_week', columns='hour_of_day', values='avg_tti')

fig = go.Figure(data=go.Heatmap(
        z=pivot_df.values,
        x=pivot_df.columns,
        y=[days[i-1] for i in pivot_df.index],
        colorscale='Viridis',
        hoverongaps=False))

fig.update_layout(
    title='Average Congestion Heatmap (Last 30 Days)',
    xaxis_title='Hour of Day (UTC)',
    yaxis_title='Day of Week')

fig.show()

## 6. Audit Table Partitions
To manage costs and understand data retention, you can audit the physical partitions of your RMI tables using the `INFORMATION_SCHEMA`.

In [ ]:
%%bigquery --project $PROJECT_ID$PROJECT_ID
SELECT
  table_schema as dataset,
  table_name as table, 
  partition_id, 
  total_rows, 
  total_logical_bytes, 
  total_billable_bytes, 
  last_modified_time, 
  storage_tier
FROM
  `historical_roads_data.INFORMATION_SCHEMA.PARTITIONS`
ORDER BY 1, 2, 3

## 7. Advanced Partition Pruning
For maximum efficiency, you can dynamically identify the latest available partition before running your main query. This is the most cost-effective way to audit the freshest RMI data.


In [ ]:
%%bigquery --project $PROJECT_ID
DECLARE last_partition TIMESTAMP;

SET last_partition = (
  SELECT
    PARSE_TIMESTAMP("%Y%m%d", MAX(partition_id))
  FROM
    `historical_roads_data.INFORMATION_SCHEMA.PARTITIONS`
  WHERE
    table_name = "historical_travel_time"
);

SELECT
  *
FROM
  `historical_roads_data.historical_travel_time`
WHERE
  record_time >= last_partition
LIMIT 10


## Summary & Next Steps
You have successfully verified your RMI data in BigQuery using native magics.

**Next**: In the next tutorial, we will explore real-time data flow via **Pub/Sub**.


---
**For more advanced RMI query patterns, visit the official [RMI Sample Queries Repository](https://github.com/googlemaps-samples/insights-samples/tree/main/roads_management_insights/rmi_sample_queries).**